# 11. Prompting Example — 구조물 점검 보고서 생성

**목표**: Meal plan 예제(05~09)에서 배운 4가지 프롬프트 엔지니어링 기법을  
**새로운 도메인(건축공학)**에 적용하여 프롬프트를 점진적으로 개선합니다.

| 단계 | 기법 | Cell | 목표 점수 |
| --- | --- | --- | --- |
| v1 | Baseline (모호한 프롬프트) | Cell 6a | ~2–3 |
| v2 | Clear & Direct | Cell 6b | ~4–5 |
| v3 | Being Specific (Guidelines) | Cell 6c | ~7–8 |
| v4 | XML Tags + Few-Shot | Cell 6d | ~9+ |

---

## 실습 방법

1. **Cell 1~4**: 공통 설정 (수정 없이 실행)
2. **Cell 5**: 데이터셋 생성 (수정 없이 실행)
3. **Cell 6a~6d**: 각 단계별 프롬프트를 직접 작성 → 실행 → 점수 확인
4. **Cell 7**: 평가 실행 (Cell 6 중 하나를 선택해 최종 평가)

> **Tip**: Cell 6a~6d를 순서대로 작성하면서 점수 변화를 관찰하세요.

In [ ]:
# Cell 1: Imports
import json
import concurrent.futures
import re
from textwrap import dedent
from statistics import mean
from dotenv import load_dotenv
from anthropic import Anthropic

In [ ]:
# Cell 2: Client 초기화 및 헬퍼 함수

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Cell 3: Report Builder (09_prompting_completed 와 동일)
def generate_prompt_evaluation_report(evaluation_results):
    total_tests = len(evaluation_results)
    scores = [result["score"] for result in evaluation_results]
    avg_score = mean(scores) if scores else 0
    max_possible_score = 10
    pass_rate = (
        100 * len([s for s in scores if s >= 7]) / total_tests if total_tests else 0
    )

    html = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Prompt Evaluation Report</title>
        <style>
            body {{ font-family: Arial, sans-serif; line-height: 1.6; padding: 20px; color: #333; }}
            .header {{ background-color: #f0f0f0; padding: 20px; border-radius: 5px; margin-bottom: 20px; }}
            .summary-stats {{ display: flex; justify-content: space-between; flex-wrap: wrap; gap: 10px; }}
            .stat-box {{ background-color: #fff; border-radius: 5px; padding: 15px; box-shadow: 0 2px 5px rgba(0,0,0,0.1); flex-basis: 30%; min-width: 200px; }}
            .stat-value {{ font-size: 24px; font-weight: bold; margin-top: 5px; }}
            table {{ width: 100%; border-collapse: collapse; margin-top: 20px; }}
            th {{ background-color: #4a4a4a; color: white; text-align: left; padding: 12px; }}
            td {{ padding: 10px; border-bottom: 1px solid #ddd; vertical-align: top; width: 20%; }}
            tr:nth-child(even) {{ background-color: #f9f9f9; }}
            .score {{ font-weight: bold; padding: 5px 10px; border-radius: 3px; display: inline-block; }}
            .score-high {{ background-color: #c8e6c9; color: #2e7d32; }}
            .score-medium {{ background-color: #fff9c4; color: #f57f17; }}
            .score-low {{ background-color: #ffcdd2; color: #c62828; }}
            .output pre {{ background-color: #f5f5f5; border: 1px solid #ddd; border-radius: 4px; padding: 10px; font-family: monospace; font-size: 14px; white-space: pre-wrap; word-wrap: break-word; }}
            .score-col {{ width: 80px; }}
        </style>
    </head>
    <body>
        <div class="header">
            <h1>Prompt Evaluation Report</h1>
            <div class="summary-stats">
                <div class="stat-box"><div>Total Test Cases</div><div class="stat-value">{total_tests}</div></div>
                <div class="stat-box"><div>Average Score</div><div class="stat-value">{avg_score:.1f} / {max_possible_score}</div></div>
                <div class="stat-box"><div>Pass Rate (>=7)</div><div class="stat-value">{pass_rate:.1f}%</div></div>
            </div>
        </div>
        <table><thead><tr><th>Scenario</th><th>Prompt Inputs</th><th>Solution Criteria</th><th>Output</th><th>Score</th><th>Reasoning</th></tr></thead><tbody>
    """

    for result in evaluation_results:
        prompt_inputs_html = "<br>".join(
            [f"<strong>{key}:</strong> {value}" for key, value in result["test_case"]["prompt_inputs"].items()]
        )
        criteria_string = "<br>\u2022 ".join(result["test_case"]["solution_criteria"])
        score = result["score"]
        score_class = "score-high" if score >= 8 else ("score-low" if score <= 5 else "score-medium")

        html += f"""
            <tr>
                <td>{result["test_case"]["scenario"]}</td>
                <td>{prompt_inputs_html}</td>
                <td>\u2022 {criteria_string}</td>
                <td class="output"><pre>{result["output"]}</pre></td>
                <td class="score-col"><span class="score {score_class}">{score}</span></td>
                <td>{result["reasoning"]}</td>
            </tr>
        """

    html += "</tbody></table></body></html>"
    return html

In [ ]:
# Cell 4: PromptEvaluator 클래스 (09_prompting_completed 와 동일)
class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=3):
        self.max_concurrent_tasks = max_concurrent_tasks

    def render(self, template_string, variables):
        placeholders = re.findall(r"{([^{}]+)}", template_string)
        result = template_string
        for placeholder in placeholders:
            if placeholder in variables:
                result = result.replace("{" + placeholder + "}", str(variables[placeholder]))
        return result.replace("{{", "{").replace("}}", "}")

    def generate_unique_ideas(self, task_description, prompt_inputs_spec, num_cases):
        prompt = """
        Generate {num_cases} unique, diverse ideas for testing a prompt that accomplishes this task:
        <task_description>{task_description}</task_description>
        The prompt will receive the following inputs
        <prompt_inputs>{prompt_inputs_spec}</prompt_inputs>
        Each idea should represent a distinct scenario that tests different aspects of the task.
        Output as a JSON array of brief descriptions.
        ```json
        ["idea 1", "idea 2", ...]
        ```
        Ensure each idea is clearly distinct, relevant, specific, and solvable with no more than 400 tokens.
        Remember, only generate {num_cases} unique ideas.
        """
        system_prompt = "You are a test scenario designer specialized in creating diverse testing scenarios."
        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'\"{key}\": str # {val},'
        rendered_prompt = self.render(dedent(prompt), {"task_description": task_description, "num_cases": num_cases, "prompt_inputs": example_prompt_inputs})
        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(messages, stop_sequences=["```"], system=system_prompt, temperature=1.0)
        return json.loads(text)

    def generate_test_case(self, task_description, idea, prompt_inputs_spec={}):
        example_prompt_inputs = ""
        for key, value in prompt_inputs_spec.items():
            val = value.replace("\n", "\\n")
            example_prompt_inputs += f'\"{key}\": \"EXAMPLE_VALUE\", // {val}\n'
        allowed_keys = ", ".join([f'\"{key}\"' for key in prompt_inputs_spec.keys()])
        prompt = """
        Generate a single detailed test case for a prompt evaluation based on:
        <task_description>{task_description}</task_description>
        <specific_idea>{idea}</specific_idea>
        <allowed_input_keys>{allowed_keys}</allowed_input_keys>
        Output Format:
        ```json
        {{
            "prompt_inputs": {{ {example_prompt_inputs} }},
            "solution_criteria": ["criterion 1", "criterion 2"]
        }}
        ```
        IMPORTANT: Only use the exact input keys listed. Make it realistic. Keep criteria concise and directly tied to the task.
        """
        system_prompt = "You are a test case creator specializing in designing evaluation scenarios."
        rendered_prompt = self.render(dedent(prompt), {"allowed_keys": allowed_keys, "task_description": task_description, "idea": idea, "example_prompt_inputs": example_prompt_inputs})
        messages = []
        add_user_message(messages, rendered_prompt)
        add_assistant_message(messages, "```json")
        text = chat(messages, stop_sequences=["```"], system=system_prompt, temperature=0.7)
        test_case = json.loads(text)
        test_case["task_description"] = task_description
        test_case["scenario"] = idea
        return test_case

    def generate_dataset(self, task_description, prompt_inputs_spec={}, num_cases=1, output_file="dataset.json"):
        ideas = self.generate_unique_ideas(task_description, prompt_inputs_spec, num_cases)
        dataset = []
        completed = 0
        total = len(ideas)
        last_reported_percentage = 0
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_concurrent_tasks) as executor:
            future_to_idea = {executor.submit(self.generate_test_case, task_description, idea, prompt_inputs_spec): idea for idea in ideas}
            for future in concurrent.futures.as_completed(future_to_idea):
                try:
                    result = future.result()
                    completed += 1
                    current_percentage = int((completed / total) * 100)
                    milestone_percentage = (current_percentage // 20) * 20
                    if milestone_percentage > last_reported_percentage:
                        print(f"Generated {completed}/{total} test cases")
                        last_reported_percentage = milestone_percentage
                    dataset.append(result)
                except Exception as e:
                    print(f"Error generating test case: {e}")
        with open(output_file, "w") as f:
            json.dump(dataset, f, indent=2)
        return dataset

    def grade_output(self, test_case, output, extra_criteria):
        prompt_inputs = ""
        for key, value in test_case["prompt_inputs"].items():
            val = value.replace("\n", "\\n")
            prompt_inputs += f'\"{key}\":\"{val}\",\n'
        extra_criteria_section = ""
        if extra_criteria:
            extra_criteria_section = self.render(dedent("""
            Mandatory Requirements - ANY VIOLATION MEANS AUTOMATIC FAILURE (score of 3 or lower):
            <extra_important_criteria>{extra_criteria}</extra_important_criteria>
            """), {"extra_criteria": extra_criteria})
        eval_template = """
        Your task is to evaluate the following AI-generated solution with EXTREME RIGOR.
        Original task description: <task_description>{task_description}</task_description>
        Original task inputs: <task_inputs>{{ {prompt_inputs} }}</task_inputs>
        Solution to Evaluate: <solution>{output}</solution>
        Criteria: <criteria>{solution_criteria}</criteria>
        {extra_criteria_section}
        Scoring: 1-3 fails mandatory, 4-6 meets mandatory but has deficiencies, 7-8 meets most criteria, 9-10 meets all.
        Output as JSON: {{"strengths": string[], "weaknesses": string[], "reasoning": string, "score": number}}
        """
        eval_prompt = self.render(dedent(eval_template), {
            "task_description": test_case["task_description"],
            "prompt_inputs": prompt_inputs,
            "output": output,
            "solution_criteria": "\n".join(test_case["solution_criteria"]),
            "extra_criteria_section": extra_criteria_section,
        })
        messages = []
        add_user_message(messages, eval_prompt)
        add_assistant_message(messages, "```json")
        eval_text = chat(messages, stop_sequences=["```"], temperature=0.0)
        return json.loads(eval_text)

    def run_test_case(self, test_case, run_prompt_function, extra_criteria=None):
        output = run_prompt_function(test_case["prompt_inputs"])
        model_grade = self.grade_output(test_case, output, extra_criteria)
        return {"output": output, "test_case": test_case, "score": model_grade["score"], "reasoning": model_grade["reasoning"]}

    def run_evaluation(self, run_prompt_function, dataset_file, extra_criteria=None, json_output_file="output.json", html_output_file="output.html"):
        with open(dataset_file, "r") as f:
            dataset = json.load(f)
        results = []
        completed = 0
        total = len(dataset)
        last_reported_percentage = 0
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_concurrent_tasks) as executor:
            future_to_test_case = {executor.submit(self.run_test_case, test_case, run_prompt_function, extra_criteria): test_case for test_case in dataset}
            for future in concurrent.futures.as_completed(future_to_test_case):
                result = future.result()
                completed += 1
                current_percentage = int((completed / total) * 100)
                milestone_percentage = (current_percentage // 20) * 20
                if milestone_percentage > last_reported_percentage:
                    print(f"Graded {completed}/{total} test cases")
                    last_reported_percentage = milestone_percentage
                results.append(result)
        average_score = mean([result["score"] for result in results])
        print(f"Average score: {average_score}")
        with open(json_output_file, "w") as f:
            json.dump(results, f, indent=2)
        html = generate_prompt_evaluation_report(results)
        with open(html_output_file, "w", encoding="utf-8") as f:
            f.write(html)
        return results


# 인스턴스 생성
evaluator = PromptEvaluator(max_concurrent_tasks=1)

In [ ]:
# Cell 5: 데이터셋 생성 — 구조물 점검 보고서
#
# task_description: 프롬프트가 수행할 작업을 설명
# prompt_inputs_spec: 프롬프트에 전달할 변수들

dataset = evaluator.generate_dataset(
    task_description="Write a concise structural inspection summary report for a building",
    prompt_inputs_spec={
        "building_type": "Type of building (e.g., RC apartment, steel office, masonry warehouse)",
        "age_years": "Age of the building in years",
        "num_floors": "Number of floors (including basement if any)",
        "observed_issues": "List of observed structural issues during inspection",
    },
    output_file="dataset_inspection.json",
    num_cases=3,
)

---

## v1: Baseline — 모호한 프롬프트

아무런 기법 없이, 가장 단순한 프롬프트로 시작합니다.  
**예상 점수**: ~2–3점

In [ ]:
# Cell 6a: v1 Baseline — 모호한 프롬프트
#
# 이 프롬프트는 의도적으로 모호합니다.
# 실행 후 점수를 확인하고, 왜 낮은지 생각해보세요.

def run_prompt(prompt_inputs):
    prompt = f"""What's wrong with this building? {prompt_inputs['observed_issues']}"""

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


# 단일 테스트 실행으로 빠르게 점수 확인
with open("dataset_inspection.json", "r") as f:
    _ds = json.load(f)
result = evaluator.run_test_case(_ds[0], run_prompt)
print(f"Score: {result['score']}")
print(f"Output preview: {result['output'][:300]}...")

---

## v2: Clear & Direct — 명확하고 직접적으로

**적용 기법**: 동작 동사로 시작, 구체적 역할 부여, 원하는 출력 명시  
**예상 점수**: ~4–5점

> **힌트**: "What's wrong?" 대신 "Generate a structural inspection report for..." 형태로 변경

In [ ]:
# Cell 6b: v2 Clear & Direct
#
# TODO: 아래 프롬프트를 Clear & Direct 원칙에 맞게 수정하세요.
# - 동작 동사(Generate, Write, Create)로 시작
# - 역할(structural engineer)을 명시
# - 모든 입력 변수를 활용

def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a structural inspection summary report for a {prompt_inputs['building_type']}.
    The building is {prompt_inputs['age_years']} years old with {prompt_inputs['num_floors']} floors.
    Observed issues: {prompt_inputs['observed_issues']}
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


result = evaluator.run_test_case(_ds[0], run_prompt)
print(f"Score: {result['score']}")
print(f"Output preview: {result['output'][:300]}...")

---

## v3: Being Specific — 가이드라인 추가

**적용 기법**: 출력 형식, 포함 요소, 처리 단계를 구체적으로 명시  
**예상 점수**: ~7–8점

> **힌트**: Guidelines 항목을 추가하세요 — 예: 심각도 등급, 보수 권고, 안전 판정 등

In [ ]:
# Cell 6c: v3 Being Specific
#
# TODO: v2 프롬프트에 Guidelines를 추가하세요.
# 아래 항목 중 필요한 것을 선택하거나 직접 작성하세요:
# 1. 심각도를 Critical / Major / Minor로 분류
# 2. 각 이슈에 대한 보수/보강 권고 포함
# 3. 전체 안전 등급을 A(양호)~E(위험)로 판정
# 4. 긴급 조치가 필요한 항목 별도 표시
# 5. 보고서 길이를 400자 이내로 유지
# 6. 건물 유형에 적합한 구조 시스템 언급

def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a structural inspection summary report for a {prompt_inputs['building_type']}.
    The building is {prompt_inputs['age_years']} years old with {prompt_inputs['num_floors']} floors.
    Observed issues: {prompt_inputs['observed_issues']}

    Guidelines:
    1. Classify each issue by severity: Critical, Major, or Minor
    2. Include repair/retrofit recommendations for each issue
    3. Assign an overall safety grade from A (Good) to E (Dangerous)
    4. Flag any items requiring immediate action
    5. Keep the report concise (under 400 words)
    6. Reference the structural system appropriate for the building type
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


result = evaluator.run_test_case(_ds[0], run_prompt)
print(f"Score: {result['score']}")
print(f"Output preview: {result['output'][:500]}...")

---

## v4: XML Tags + Few-Shot — 최종 프롬프트

**적용 기법**:  
- `<building_info>` 태그로 입력 구조화  
- `<guidelines>` 태그로 지침 분리  
- `<sample_input>` / `<ideal_output>` 으로 Few-Shot 예시 제공  

**예상 점수**: ~9+점

In [ ]:
# Cell 6d: v4 XML Tags + Few-Shot (완성 예시)
#
# TODO: 아래 프롬프트를 참고하여 XML 태그와 Few-Shot 예시를 추가하세요.
# 직접 수정하거나, 이 셀을 그대로 실행하여 최종 점수를 확인할 수 있습니다.

def run_prompt(prompt_inputs):
    prompt = f"""
    Generate a structural inspection summary report for a building based on the following information.

    <building_info>
    - Building type: {prompt_inputs['building_type']}
    - Age: {prompt_inputs['age_years']} years
    - Floors: {prompt_inputs['num_floors']}
    - Observed issues: {prompt_inputs['observed_issues']}
    </building_info>

    <guidelines>
    1. Classify each issue by severity: Critical, Major, or Minor
    2. Include specific repair/retrofit recommendations for each issue
    3. Assign an overall safety grade: A (Good) / B (Fair) / C (Poor) / D (Serious) / E (Dangerous)
    4. Flag items requiring immediate action with [URGENT]
    5. Keep the report concise (under 400 words)
    6. Reference the structural system appropriate for the building type
    </guidelines>

    Here is an example with a sample input and an ideal output:

    <sample_input>
    Building type: RC apartment
    Age: 35 years
    Floors: 15F + B2
    Observed issues: Concrete spalling at B1 columns, hairline cracks in 7F shear wall, corroded rebar exposed at rooftop parapet
    </sample_input>

    <ideal_output>
    ## Structural Inspection Summary

    **Building**: RC apartment, 15F + B2, 35 years old
    **Structural System**: Reinforced concrete shear wall + frame system
    **Overall Safety Grade**: C (Poor)

    ### Issue Assessment

    | # | Location | Issue | Severity | Recommendation |
    |---|----------|-------|----------|----------------|
    | 1 | B1 columns | Concrete spalling | **Critical** [URGENT] | Immediate shoring; epoxy injection and FRP wrapping within 30 days |
    | 2 | 7F shear wall | Hairline cracks (< 0.3mm) | Minor | Monitor crack width quarterly; seal with flexible sealant if widening |
    | 3 | Rooftop parapet | Corroded rebar exposed | Major | Remove deteriorated concrete, treat rebar with rust inhibitor, patch with repair mortar |

    ### Immediate Actions Required
    - [URGENT] B1 column spalling: Install temporary steel shores and restrict parking in affected bays
    - Schedule detailed structural assessment within 2 weeks

    ### Summary
    The B1 column deterioration poses the highest risk as it affects the primary load path. The shear wall cracks are non-structural at this stage but require monitoring. The parapet corrosion is a maintenance issue that should be addressed within 3 months to prevent further section loss.
    </ideal_output>

    This example shows well-structured output with clear severity classification, actionable recommendations, and a practical summary.
    """

    messages = []
    add_user_message(messages, prompt)
    return chat(messages)


result = evaluator.run_test_case(_ds[0], run_prompt)
print(f"Score: {result['score']}")
print(f"Reasoning: {result['reasoning']}")
print(f"\n--- Output ---\n{result['output'][:800]}")

---

## 최종 평가 실행

위 Cell 6a~6d 중 가장 만족스러운 버전의 `run_prompt`를 선택한 뒤,  
아래 셀을 실행하여 전체 데이터셋에 대한 종합 평가를 수행합니다.

In [ ]:
# Cell 7: 전체 데이터셋 평가 실행
#
# 위에서 정의한 run_prompt 중 마지막 실행한 버전이 사용됩니다.
# 특정 버전을 사용하려면 해당 Cell 6x를 먼저 다시 실행하세요.

results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset_inspection.json",
    extra_criteria="""
    The output should include:
    - Overall safety grade (A through E)
    - Severity classification for each issue (Critical/Major/Minor)
    - Specific repair recommendations
    - Urgent items clearly flagged
    """,
)

---

## 도전 과제 (선택)

1. **System Prompt 추가**: `chat()` 함수의 `system` 파라미터를 활용하여  
   "You are a licensed structural engineer with 20 years of experience." 같은 역할을 부여해보세요.

2. **한국어 보고서**: 프롬프트를 한국어로 작성하고, 한국어 보고서가 생성되는지 확인해보세요.  
   한국 건축법 기준(KDS 41 등)을 참조하도록 유도할 수 있습니다.

3. **Code Grading 추가**: `run_test_case`를 오버라이드하여  
   "Safety Grade"가 A~E 중 하나인지 코드로 검증하는 채점기를 추가해보세요.

4. **나만의 도메인**: `task_description`과 `prompt_inputs_spec`을 완전히 바꿔서  
   본인의 연구 분야에 맞는 평가 파이프라인을 구축해보세요.